# 13. Error analysis Span NER: BIN и ACT

Анализ выполняется на validation с глобальным threshold, ранее выбранным по micro-F1. Test не используется. Результаты сохраняются в небольшие CSV/JSON-файлы.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import runpy

PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
EXPERIMENT_CONFIG = PROJECT_DIR / 'configs/experiments/span_ner_corrected_v1.yaml'
OUTPUT_DIR = PROJECT_DIR / 'results/span_ner_corrected_v1/seed_42'
CHECKPOINT = OUTPUT_DIR / 'checkpoints/best'
CALIBRATION = OUTPUT_DIR / 'threshold_calibration.json'
BOOTSTRAP = PROJECT_DIR / 'colab_bootstrap.py'

for required_path in (EXPERIMENT_CONFIG, CHECKPOINT, CALIBRATION, BOOTSTRAP):
    if not required_path.exists():
        raise FileNotFoundError(f'Не найден {required_path}. Выполните ноутбуки 10 и 11.')
GLOBAL_THRESHOLD = float(json.loads(CALIBRATION.read_text(encoding='utf-8'))['best_threshold'])
bootstrap_project = runpy.run_path(str(BOOTSTRAP))['bootstrap_project']
bootstrap_project(PROJECT_DIR)
print('Global threshold:', GLOBAL_THRESHOLD)

In [ ]:
from rurebus_ie.training import run_span_validation_error_analysis

analysis = run_span_validation_error_analysis(
    EXPERIMENT_CONFIG,
    project_root=PROJECT_DIR,
    checkpoint_dir=CHECKPOINT,
    confidence_threshold=GLOBAL_THRESHOLD,
    artifact_name='global_threshold',
)
analysis.summary

In [ ]:
import pandas as pd

errors = pd.DataFrame(analysis.errors)
per_class = pd.DataFrame(analysis.per_class).set_index('entity_type')
confusion = pd.DataFrame(analysis.confusion)
display(per_class.sort_values('f1'))
display(pd.DataFrame(analysis.length_breakdown))

## Срез ошибок BIN и ACT

In [ ]:
FOCUS_TYPES = ['BIN', 'ACT']
focus = errors[
    errors['gold_type'].isin(FOCUS_TYPES) | errors['predicted_type'].isin(FOCUS_TYPES)
].copy()
display(
    focus.groupby(['category', 'gold_type', 'predicted_type'], dropna=False)
         .size().rename('count').sort_values(ascending=False).head(30).reset_index()
)

In [ ]:
bin_false_positive = errors[
    (errors['predicted_type'] == 'BIN') & (errors['category'] != 'true_positive')
]
act_false_negative = errors[
    (errors['gold_type'] == 'ACT') & (errors['category'] != 'true_positive')
]

print('Частые ложные BIN:')
display(bin_false_positive['predicted_text'].value_counts().head(30).rename('count').to_frame())
print('Часто пропускаемые ACT:')
display(act_false_negative['gold_text'].value_counts().head(30).rename('count').to_frame())

In [ ]:
columns = [
    'document_id', 'category', 'gold_type', 'predicted_type',
    'gold_text', 'predicted_text', 'confidence', 'iou',
    'start_correct', 'end_correct', 'near_window_edge'
]
display(focus[columns].head(100))
print('Полный отчёт:', OUTPUT_DIR / 'error_analysis/validation/global_threshold')